In [2]:
"""
00_00_batch_amp_diagnostic_config.txt
Phase 2 - P3 -- Section 0: Reusable GPU Capacity Diagnostic
Shared configuration for the batch-size / AMP capacity check. Re-run this
whenever hardware, resolution, or architecture changes -- this is a
reusable utility, not a one-time script.
"""

import torch

# --- Candidates to test, in increasing order ---
BATCH_CANDIDATES = [16, 32, 48, 64]

# --- Per-model input resolution (locked from Phase 2_Implementation.md) ---
MODEL_INPUT_SIZES = {
    "efficientnet_b3": 300,
    "resnet50": 300,
    "vit_b_16": 224,
}

NUM_CLASSES = 7  # HAM10000 taxonomy

# --- Precision under test ---
AMP_DTYPE = torch.bfloat16  # bf16, not fp16 -- fp16 overflows EfficientNet-B3's
                            # forward pass; bf16 has fp32's exponent range

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "This GPU does not support bfloat16. Re-confirm the AMP precision "
        "decision before proceeding -- do not silently fall back to fp16 or fp32."
    )

print("Batch/AMP diagnostic config loaded.")
print(f"  Device            : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU               : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM              : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"  Batch candidates  : {BATCH_CANDIDATES}")
print(f"  AMP dtype         : {AMP_DTYPE}")

Batch/AMP diagnostic config loaded.
  Device            : cuda
  GPU               : NVIDIA GeForce RTX 5060 Ti
  VRAM              : 15.9 GB
  Batch candidates  : [16, 32, 48, 64]
  AMP dtype         : torch.bfloat16


In [3]:
"""
00_01_batch_amp_diagnostic_driver.txt
Phase 2 - P3 -- Section 0: Reusable GPU Capacity Diagnostic (driver)

Run AFTER 00_00_batch_amp_diagnostic_config.txt has been executed in the
same notebook/session (this cell assumes BATCH_CANDIDATES, MODEL_INPUT_SIZES,
NUM_CLASSES, AMP_DTYPE, DEVICE already exist in memory).

What this does, per architecture:
  - builds the real model (torchvision, ImageNet-pretrained, matching the
    locked head choice is NOT relevant here -- this diagnostic only measures
    memory/precision behavior, so a plain Linear head is used regardless of
    what Cluster 3 eventually decides for the real training scripts)
  - for each candidate batch size (ascending): runs ONE real forward pass +
    backward pass under bf16 autocast, on random tensors shaped like real
    input (correct resolution, 3 channels) and random integer labels
  - records peak CUDA memory allocated for that batch size
  - confirms the loss value is finite (not NaN/Inf) under bf16
  - stops testing LARGER batch sizes for that model as soon as one OOMs
    (smaller candidates already tested for that model remain valid results)
  - frees memory between every single test (del tensors + empty_cache)
    so one model/batch-size's measurement can't pollute the next

Output: a results table (one row per model x batch size actually tested),
printed at the end. Nothing is written to disk by this cell -- it's a
one-time hardware check, not part of the training pipeline itself.
"""

import gc
from typing import cast

import torch
import torch.nn as nn
import torchvision.models as tvm

def build_model(name, num_classes):
    if name == "efficientnet_b3":
        m = tvm.efficientnet_b3(weights=tvm.EfficientNet_B3_Weights.IMAGENET1K_V1)
        old_head = cast(nn.Linear, m.classifier[-1])
        m.classifier[-1] = nn.Linear(old_head.in_features, num_classes)
    elif name == "resnet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        old_head = cast(nn.Linear, m.fc)
        m.fc = nn.Linear(old_head.in_features, num_classes)
    elif name == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1)
        old_head = cast(nn.Linear, m.heads.head)
        m.heads.head = nn.Linear(old_head.in_features, num_classes)
    else:
        raise ValueError(f"Unknown model name: {name}")
    return m


def test_one(model_name, resolution, batch_size):
    """Runs one forward+backward pass. Returns (peak_mem_gb, loss_value) or
    raises torch.cuda.OutOfMemoryError on OOM (caller handles it)."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = build_model(model_name, NUM_CLASSES).to(DEVICE)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()

    x = torch.randn(batch_size, 3, resolution, resolution, device=DEVICE)
    y = torch.randint(0, NUM_CLASSES, (batch_size,), device=DEVICE)

    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        logits = model(x)
        loss = criterion(logits, y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    peak_mem_gb = torch.cuda.max_memory_allocated() / 1024**3
    loss_value = loss.item()

    del model, optimizer, x, y, logits, loss
    gc.collect()
    torch.cuda.empty_cache()

    return peak_mem_gb, loss_value


results = []  # each row: dict(model, batch_size, peak_mem_gb, loss, finite, status)

for model_name, resolution in MODEL_INPUT_SIZES.items():
    print(f"\n=== {model_name} @ {resolution}x{resolution} ===")
    for batch_size in BATCH_CANDIDATES:
        try:
            peak_mem_gb, loss_value = test_one(model_name, resolution, batch_size)
            is_finite = torch.isfinite(torch.tensor(loss_value)).item()
            status = "OK" if is_finite else "NON-FINITE LOSS"
            print(f"  batch={batch_size:>3}  peak_mem={peak_mem_gb:6.2f} GB  "
                  f"loss={loss_value:.4f}  [{status}]")
            results.append(dict(model=model_name, batch_size=batch_size,
                                 peak_mem_gb=round(peak_mem_gb, 2),
                                 loss=round(loss_value, 4),
                                 finite=is_finite, status=status))
        except torch.cuda.OutOfMemoryError:
            print(f"  batch={batch_size:>3}  OOM -- stopping further increases for {model_name}")
            results.append(dict(model=model_name, batch_size=batch_size,
                                 peak_mem_gb=None, loss=None,
                                 finite=None, status="OOM"))
            torch.cuda.empty_cache()
            break

print("\n=== Summary ===")
for r in results:
    print(r)


=== efficientnet_b3 @ 300x300 ===
  batch= 16  peak_mem=  2.48 GB  loss=1.9697  [OK]
  batch= 32  peak_mem=  4.86 GB  loss=1.9756  [OK]
  batch= 48  peak_mem=  7.26 GB  loss=1.9666  [OK]
  batch= 64  peak_mem=  9.61 GB  loss=1.9474  [OK]

=== resnet50 @ 300x300 ===
  batch= 16  peak_mem=  1.38 GB  loss=2.0278  [OK]
  batch= 32  peak_mem=  2.60 GB  loss=1.9912  [OK]
  batch= 48  peak_mem=  3.97 GB  loss=1.9608  [OK]
  batch= 64  peak_mem=  5.25 GB  loss=1.9535  [OK]

=== vit_b_16 @ 224x224 ===
  batch= 16  peak_mem=  1.64 GB  loss=1.8477  [OK]
  batch= 32  peak_mem=  2.69 GB  loss=1.9031  [OK]
  batch= 48  peak_mem=  3.80 GB  loss=1.9855  [OK]
  batch= 64  peak_mem=  4.86 GB  loss=1.9927  [OK]

=== Summary ===
{'model': 'efficientnet_b3', 'batch_size': 16, 'peak_mem_gb': 2.48, 'loss': 1.9697, 'finite': True, 'status': 'OK'}
{'model': 'efficientnet_b3', 'batch_size': 32, 'peak_mem_gb': 4.86, 'loss': 1.9756, 'finite': True, 'status': 'OK'}
{'model': 'efficientnet_b3', 'batch_size': 48, '

In [4]:
"""
00_02_set_all_seeds.txt
Phase 2 - P3 -- Section 0: Reusable Global Seeding Utility

Call set_all_seeds(RANDOM_SEED) at the very start of train_fold, before the
model, optimizer, or balanced sampler are constructed for that fold. This
must run at the START OF EVERY FOLD (not once at notebook start) so each
fold is independently reproducible on its own.

This replaces the P1 approach of only seeding the balanced sampler's
torch.Generator -- that left Python's random module, NumPy, and Torch's own
CPU/CUDA RNGs unseeded, so nothing outside the sampler itself (weight init
order where applicable, any torch-level randomness in augmentation, etc.)
was actually reproducible fold-to-fold.
"""

import os
import random
import numpy as np
import torch


def set_all_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


# --- Usage inside train_fold (all three scripts) ---
# def train_fold(fold_idx, ...):
#     set_all_seeds(RANDOM_SEED)   # <-- first line, before model/optimizer/sampler
#     model = build_model(...)
#     ...

In [5]:
"""
00_03_checkpoint_utils.txt
Phase 2 - P3 -- Section 0: Reusable Checkpoint Utilities (power-cut resilience)

Two checkpoint files per fold, both written atomically (tmp file + os.replace,
so a mid-write power cut leaves the previous good file intact, never a
half-written one):

  <out_dir>/fold_{fold_idx}_resume.pt
      Full training state, overwritten EVERY epoch. Used only to resume a
      fold that was interrupted mid-training.

  <out_dir>/fold_{fold_idx}_best.pt
      Best-epoch model weights only, overwritten only when validation
      AUC-ROC improves. This is the file actually used for evaluation --
      matches the spec's "best-epoch weights, not final-epoch" policy.

Usage inside train_fold (after set_all_seeds(RANDOM_SEED)):

    resume_path = os.path.join(out_dir, f"fold_{fold_idx}_resume.pt")
    best_path   = os.path.join(out_dir, f"fold_{fold_idx}_best.pt")

    state = load_resume_state(resume_path)  # load BEFORE building the sampler --
                                             # Script 1's balanced sampler needs
                                             # state["sampler_generator_state"]
                                             # at construction time (01_02)
    if state is not None:
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        restore_rng_state(state["rng_state"])
        start_epoch = state["epoch"] + 1
        best_metric = state["best_metric"]
        early_stop_counter = state["early_stop_counter"]
        print(f"Resuming fold {fold_idx} from epoch {start_epoch}")
    else:
        start_epoch = 0
        best_metric = -float("inf")
        early_stop_counter = 0

    for epoch in range(start_epoch, MAX_EPOCHS):
        ... train one epoch, compute val_auc_roc ...

        # sampler is None for Scripts 2/3 (no balancing -> no sampler-specific generator)
        sampler_generator_state = sampler.generator.get_state() if sampler is not None else None
        save_resume_state(resume_path, epoch, model, optimizer,
                           best_metric, early_stop_counter,
                           sampler_generator_state=sampler_generator_state)

        if val_auc_roc > best_metric:
            best_metric = val_auc_roc
            early_stop_counter = 0
            save_best_weights(best_path, model)
        else:
            early_stop_counter += 1
            if early_stop_counter >= PATIENCE:
                break
"""

import os
import random
import tempfile

import numpy as np
import torch


def _atomic_torch_save(obj, final_path: str):
    """Writes obj to a tmp file in the same directory, then atomically
    renames it over final_path. os.replace is atomic on both POSIX and
    Windows, so a crash mid-write never leaves a corrupted checkpoint --
    the old file stays valid until the new one is fully written."""
    final_dir = os.path.dirname(final_path) or "."
    os.makedirs(final_dir, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=final_dir, suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            torch.save(obj, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, final_path)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available() and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])


def save_resume_state(resume_path, epoch, model, optimizer,
                       best_metric, early_stop_counter,
                       sampler_generator_state=None, finished=False):
    """sampler_generator_state: Script 1's balanced sampler (01_02) owns its
    own torch.Generator, separate from the global RNGs capture_rng_state()
    covers -- pass sampler.generator.get_state() here (None for Scripts 2/3,
    which have no sampler-specific generator) so a resumed fold continues
    the exact same draw sequence instead of silently restarting it from
    the sampler's original seed.

    finished: True once early stopping has actually fired for this fold.
    Without this, a later re-run of the same script (another power cut, a
    re-executed cell) can't tell "interrupted mid-training" apart from
    "already finished" -- both just look like an ordinary checkpoint -- so
    it silently resumes and keeps training a fold that was already done.
    """
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_metric": best_metric,
        "early_stop_counter": early_stop_counter,
        "rng_state": capture_rng_state(),
        "sampler_generator_state": sampler_generator_state,
        "finished": finished,
    }
    _atomic_torch_save(state, resume_path)


def load_resume_state(resume_path):
    if not os.path.exists(resume_path):
        return None
    return torch.load(resume_path, map_location="cpu", weights_only=False)


def save_best_weights(best_path, model):
    _atomic_torch_save({"model_state_dict": model.state_dict()}, best_path)

In [6]:
"""
00_04_preflight_verification.txt
Phase 2 - P3 -- Section 0: Preflight Verification

Run this ONCE at the start of the notebook, before Script 1/2/3 begin
training. All four checks run as a full set (every failure collected and
reported together, not stopped at the first one) -- same convention as
Phase 1's 01_dataset_verification.txt.

Checks:
  1. Image count -- every image_id in cv_fold_assignments.csv has a
     matching .jpg on disk
  2. Fold CSV integrity -- no nulls in required columns, cv_fold only
     takes values {0,1,2,3,4}
  3. Lesion-leakage re-check -- independently re-verifies no lesion_id
     spans more than one cv_fold (re-running Section 9's own check here,
     rather than trusting the Phase 1 result blindly -- this is Phase 2's
     own gate, not a copy of Phase 1's)
  4. Class-weight recomputation match -- recomputes each fold's
     effective-number weights (beta=0.999) from cv_fold_assignments.csv's
     own per-fold training counts, and confirms it matches the saved
     final_class_weights_per_fold.csv from Phase 1

Raises RuntimeError with a full summary if ANY check fails -- this is a
hard gate, not a warning.
"""

from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd

BASE_DIR = Path(r"T:\Thesis - Development")
ARTIFACTS_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2"
FINAL_IMAGES_DIR = ARTIFACTS_DIR / "Final Output Phase 1" / "images"

CV_FOLD_PATH = ARTIFACTS_DIR / "cv_fold_assignments.csv"
WEIGHTS_PATH = ARTIFACTS_DIR / "final_class_weights_per_fold.csv"

BETA_FINAL = 0.999  # locked, Phase 1 Section 10
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]


def compute_effective_number_weights(class_counts, beta):
    """Same formula as Phase 1's 10_00_effective_number_function.txt --
    duplicated here (not imported) so this preflight has no hidden
    dependency on the Phase 1 notebook still being open/run."""
    classes = list(class_counts.keys()) if isinstance(class_counts, dict) else class_counts.index.tolist()
    counts = np.array([class_counts[c] for c in classes], dtype=np.float64)
    effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    raw_weights = 1.0 / effective_num
    normalized_weights = raw_weights / raw_weights.sum() * len(classes)
    return dict(zip(classes, normalized_weights))


def run_preflight_verification():
    failures = []
    print("=" * 60)
    print("PREFLIGHT VERIFICATION -- Phase 2 P2")
    print("=" * 60)

    cv_df = pd.read_csv(CV_FOLD_PATH)

    # --- 1. Image count check ---
    print("\n[1/4] Image count check...")
    expected_ids = set(cv_df["image_id"])
    missing = [iid for iid in expected_ids if not (FINAL_IMAGES_DIR / f"{iid}.jpg").exists()]
    if missing:
        failures.append(f"Image count: {len(missing)} image_id(s) in {CV_FOLD_PATH.name} "
                         f"have no matching .jpg in {FINAL_IMAGES_DIR}. "
                         f"Sample missing: {missing[:5]}")
        print(f"  FAILED -- {len(missing)} missing image(s)")
    else:
        print(f"  OK -- all {len(expected_ids)} image_ids in the fold CSV have a matching file")

    # --- 2. Fold CSV integrity check ---
    print("\n[2/4] Fold CSV integrity check...")
    required_cols = {"image_id", "lesion_id", "dx", "localization", "cv_fold"}
    missing_cols = required_cols - set(cv_df.columns)
    if missing_cols:
        failures.append(f"Fold CSV integrity: missing column(s) {missing_cols}")
        print(f"  FAILED -- missing columns: {missing_cols}")
    else:
        null_counts = cv_df[list(required_cols)].isna().sum()
        nulls_present = null_counts[null_counts > 0]
        if len(nulls_present) > 0:
            failures.append(f"Fold CSV integrity: nulls found -- {nulls_present.to_dict()}")
            print(f"  FAILED -- nulls found: {nulls_present.to_dict()}")
        else:
            bad_folds = set(cv_df["cv_fold"].unique()) - {0, 1, 2, 3, 4}
            if bad_folds:
                failures.append(f"Fold CSV integrity: unexpected cv_fold value(s) {bad_folds}")
                print(f"  FAILED -- unexpected cv_fold values: {bad_folds}")
            else:
                print(f"  OK -- no nulls, cv_fold values are exactly {{0,1,2,3,4}}, "
                      f"{len(cv_df)} rows")

    # --- 3. Lesion-leakage re-check ---
    print("\n[3/4] Lesion-leakage re-check...")
    lesion_fold_counts = cv_df.groupby("lesion_id")["cv_fold"].nunique()
    leaked = lesion_fold_counts[lesion_fold_counts > 1]
    if len(leaked) > 0:
        failures.append(f"Lesion leakage: {len(leaked)} lesion_id(s) span more than one "
                         f"cv_fold. Sample: {leaked.index.tolist()[:5]}")
        print(f"  FAILED -- {len(leaked)} leaked lesion(s)")
    else:
        print(f"  OK -- all {cv_df['lesion_id'].nunique()} unique lesions confined to a "
              f"single fold")

    # --- 4. Class-weight recomputation match ---
    print("\n[4/4] Class-weight recomputation match...")
    if not WEIGHTS_PATH.exists():
        failures.append(f"Class-weight check: {WEIGHTS_PATH} not found")
        print(f"  FAILED -- {WEIGHTS_PATH} not found")
    else:
        saved_weights = pd.read_csv(WEIGHTS_PATH, index_col="dx")
        mismatches = []
        for fold_num in sorted(cv_df["cv_fold"].unique()):
            train_subset = cv_df[cv_df["cv_fold"] != fold_num]
            counts = train_subset["dx"].value_counts().reindex(CLASSES, fill_value=0)
            recomputed = compute_effective_number_weights(counts.to_dict(), BETA_FINAL)

            fold_col = str(fold_num)
            if fold_col not in saved_weights.columns:
                mismatches.append(f"fold {fold_num}: column '{fold_col}' not in saved weights file")
                continue

            for cls in CLASSES:
                saved_val = float(cast(float, saved_weights.loc[cls, fold_col]))
                recomputed_val = float(recomputed[cls])
                if not np.isclose(saved_val, recomputed_val, rtol=1e-4):
                    mismatches.append(
                        f"fold {fold_num}, class {cls}: saved={saved_val:.4f} "
                        f"vs recomputed={recomputed_val:.4f}"
                    )

        if mismatches:
            failures.append(f"Class-weight recomputation mismatch(es): {mismatches}")
            print(f"  FAILED -- {len(mismatches)} mismatch(es), e.g. {mismatches[:3]}")
        else:
            print(f"  OK -- recomputed weights match {WEIGHTS_PATH.name} for all 5 folds, "
                  f"all 7 classes (rtol=1e-4)")

    print("\n" + "=" * 60)
    if failures:
        print(f"PREFLIGHT FAILED -- {len(failures)} issue(s) found:")
        for f in failures:
            print(f"  - {f}")
        print("=" * 60)
        raise RuntimeError(
            f"Preflight verification failed with {len(failures)} issue(s). "
            f"Training must not start until these are resolved. See printed detail above."
        )
    else:
        print("PREFLIGHT PASSED -- all 4 checks OK. Safe to proceed to training.")
        print("=" * 60)


# --- Usage ---
# run_preflight_verification()   # <-- call this once, before Script 1 starts

In [7]:
"""
00_05_save_run_config.txt
Phase 2 - P3 -- Section 0: run_config.json writer (Final Gate input)

Each of the three scripts calls save_run_config() ONCE, after its own
config is fully set (not per-fold -- these are script-level settings,
identical across all 5 folds). The Final Gate cell (00_07) reads these
three files back and checks them against what the spec locks.

Written atomically (reuses 00_03's _atomic_torch_save pattern, adapted
for JSON) so a power cut mid-write can't leave a corrupt config file.

NOTE: this file uses PHASE2_ARTIFACTS_DIR, not ARTIFACTS_DIR -- 01_00
(Dataset) defines a global named ARTIFACTS_DIR pointing at the PHASE 1
folder (correct there -- that's where images/fold CSVs live), and since
every cell shares one kernel namespace, a same-named variable here would
get silently overwritten by whichever cell ran last. Distinct name,
no collision, no silent bug.
"""

import json
import os
import tempfile
from pathlib import Path

PHASE2_ARTIFACTS_DIR = Path(r"T:\Thesis - Development") / "Resources" / "Artifacts" / "Phase 2 - P3"
PHASE2_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def save_run_config(script_name: str, config: dict):
    """
    script_name: one of "script1_efficientnet_b3", "script2_resnet50",
                 "script3_vit_b_16" -- fixes the output filename.
    config: the full run_config dict -- see the schema comment in each
            script's own call site for the exact keys expected.
    """
    out_path = PHASE2_ARTIFACTS_DIR / f"{script_name}_run_config.json"
    tmp_fd, tmp_path = tempfile.mkstemp(dir=PHASE2_ARTIFACTS_DIR, suffix=".tmp")
    try:
        with os.fdopen(tmp_fd, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, out_path)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise
    print(f"run_config saved: {out_path}")


# --- Usage, e.g. at the end of Script 1's config cell ---
# save_run_config("script1_efficientnet_b3", {
#     "model": "efficientnet_b3",
#     "resolution": 300,
#     "balancing_enabled": True,
#     "batch_size": 32,
#     "optimizer": "AdamW",
#     "weight_decay": 1e-5,
#     "learning_rate": 1e-4,
#     "early_stopping_patience": 10,
#     "max_epochs": 100,
#     "checkpoint_policy": "best_epoch_auc_roc",
#     "monitor_metric": "val_auc_roc",
#     "amp_dtype": "bfloat16",
#     "random_seed": 42,
# })

In [8]:
"""
00_06_mechanism_ab_verification.txt
Phase 2 - P3 -- Section 0: Class-Balancing Independent Verification
(Script 1 / EfficientNet-B3 ONLY -- Script 2/3 use no balancing at all)

Per the spec's "Verify Before Moving On": loss weights and the balanced
sampler must be confirmed independently active, not one used as a proxy
for the other. Two mechanisms, genuinely separate code paths:

  Mechanism A -- confirms the weight TENSOR ACTUALLY WIRED INTO THE LOSS
  FUNCTION for a given fold matches the expected effective-number weights,
  in the correct class order. This catches a real class of bug the
  preflight (00_04) can't: preflight only checks the saved CSV against a
  recomputation -- it never touches the live criterion object. A bug in
  how the weights tensor gets built or ordered when constructing
  nn.CrossEntropyLoss would pass preflight but silently train wrong.

  Mechanism B -- confirms the balanced SAMPLER is genuinely active inside
  the real, wired-up training DataLoader (not a standalone reproduction).
  Runs ~300 batches, tallies observed class frequency, checks it lands
  near the ~14.29%-per-class target. Mirrors Phase 1's 10_05 diagnostic,
  but run here against Script 1's actual DataLoader object.

Call both once per fold, right after that fold's model/criterion/
DataLoader are built, before training starts.
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
BETA_FINAL = 0.999


def compute_effective_number_weights(class_counts, beta):
    classes = list(class_counts.keys()) if isinstance(class_counts, dict) else class_counts.index.tolist()
    counts = np.array([class_counts[c] for c in classes], dtype=np.float64)
    effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    raw_weights = 1.0 / effective_num
    normalized_weights = raw_weights / raw_weights.sum() * len(classes)
    return dict(zip(classes, normalized_weights))


def verify_mechanism_a(criterion: nn.CrossEntropyLoss, cv_df: pd.DataFrame, fold_num: int,
                        class_to_idx: dict, rtol: float = 1e-4):
    """
    criterion: the actual nn.CrossEntropyLoss instance built for this fold
               (must have been constructed with weight=<tensor>).
    cv_df: the loaded cv_fold_assignments.csv DataFrame.
    fold_num: which fold this criterion belongs to.
    class_to_idx: {class_name: logit_index} -- the SAME mapping used to
                  build the model's output ordering. Mechanism A checks
                  the weight tensor against expected values IN THIS ORDER,
                  since a class-order mismatch here is exactly the kind of
                  bug this mechanism exists to catch.
    """
    print(f"--- Mechanism A: loss-weight wiring check, fold {fold_num} ---")

    if criterion.weight is None:
        raise AssertionError(
            f"Fold {fold_num}: criterion.weight is None -- class-balanced loss is "
            f"NOT active. Script 1 requires weighted CrossEntropyLoss."
        )

    train_subset = cv_df[cv_df["cv_fold"] != fold_num]
    counts = train_subset["dx"].value_counts().reindex(CLASSES, fill_value=0)
    expected = compute_effective_number_weights(counts.to_dict(), BETA_FINAL)

    wired_weights = criterion.weight.detach().cpu().numpy()
    mismatches = []
    for cls, idx in class_to_idx.items():
        wired_val = float(wired_weights[idx])
        expected_val = float(expected[cls])
        if not np.isclose(wired_val, expected_val, rtol=rtol):
            mismatches.append(f"class {cls} (idx {idx}): wired={wired_val:.4f} vs expected={expected_val:.4f}")

    if mismatches:
        raise AssertionError(f"Fold {fold_num}: Mechanism A FAILED -- {mismatches}")
    print(f"  OK -- criterion.weight matches expected effective-number weights, "
          f"correct class order, all {len(CLASSES)} classes")


def verify_mechanism_b(train_loader: DataLoader, n_batches: int = 300, tolerance_pp: float = 3.0):
    """
    train_loader: the ACTUAL DataLoader Script 1 will train with (must
                  already have the balanced sampler attached).
    n_batches: how many batches to sample for the empirical tally.
    tolerance_pp: max allowed deviation (percentage points) from the
                  ~14.29% ideal target, per class, before this fails.
    """
    n_classes = len(CLASSES)
    ideal_target_pct = 100.0 / n_classes
    print(f"--- Mechanism B: balanced-sampler wiring check ({n_batches} batches) ---")

    observed_labels = []
    for batch_num, batch in enumerate(train_loader):
        _, labels = batch[0], batch[1]
        observed_labels.extend(labels.tolist() if torch.is_tensor(labels) else list(labels))
        if batch_num + 1 >= n_batches:
            break

    observed_series = pd.Series(observed_labels)
    empirical_pct = (observed_series.value_counts(normalize=True) * 100)

    print(f"  Ideal balanced target: {ideal_target_pct:.2f}% per class")
    print(empirical_pct.round(2).to_string())

    max_deviation = (empirical_pct - ideal_target_pct).abs().max()
    if max_deviation > tolerance_pp:
        raise AssertionError(
            f"Mechanism B FAILED -- max deviation from ideal target is "
            f"{max_deviation:.2f}pp, exceeds tolerance of {tolerance_pp}pp. "
            f"Balanced sampler may not be genuinely active."
        )
    print(f"  OK -- max deviation from ideal target: {max_deviation:.2f}pp "
          f"(tolerance: {tolerance_pp}pp)")


# --- Usage, per fold, right after model/criterion/train_loader are built ---
# verify_mechanism_a(criterion, cv_df, fold_num, class_to_idx)
# verify_mechanism_b(train_loader)

In [9]:
"""
00_07_final_gate.txt
Phase 2 - P3 -- Section 0: Final Gate (all three scripts)

Run this ONCE, after all three scripts have finished all 5 folds each.
Reads back each script's run_config.json (written by 00_05's
save_run_config()) and programmatically checks it against what
Phase 2_Implementation.md actually locks -- not a re-run of training,
just a paper-trail check that what ran matches what was specified.

This is the spec's own checklist item: "Confirm each model trained under
its correct pipeline" + "Confirm no further tuning happens on any of the
three after this point." Fails loudly (RuntimeError) if anything doesn't
match -- these numbers become the fixed Phase 6 reference, so a silent
mismatch here would be far more costly to catch later.

NOTE: uses PHASE2_ARTIFACTS_DIR (same distinct name as 00_05), not
ARTIFACTS_DIR -- 01_00 (Dataset) defines a global ARTIFACTS_DIR pointing
at the PHASE 1 folder, and every cell shares one kernel namespace, so a
same-named variable here would silently pick up the wrong value.
"""

import json
from pathlib import Path

PHASE2_ARTIFACTS_DIR = Path(r"T:\Thesis - Development") / "Resources" / "Artifacts" / "Phase 2 - P3"

# --- What the spec locks, per script ---
EXPECTED = {
    "script1_efficientnet_b3": {
        "model": "efficientnet_b3",
        "resolution": 300,
        "balancing_enabled": True,
        "optimizer": "AdamW",
        "weight_decay": 1e-5,
        "checkpoint_policy": "best_epoch_auc_roc",
        "monitor_metric": "val_auc_roc",
    },
    "script2_resnet50": {
        "model": "resnet50",
        "resolution": 300,
        "balancing_enabled": False,
        "optimizer": "AdamW",
        "weight_decay": 1e-5,
        "checkpoint_policy": "best_epoch_auc_roc",
        "monitor_metric": "val_auc_roc",
    },
    "script3_vit_b_16": {
        "model": "vit_b_16",
        "resolution": 224,
        "balancing_enabled": False,
        "optimizer": "AdamW",
        "weight_decay": 1e-5,
        "checkpoint_policy": "best_epoch_auc_roc",
        "monitor_metric": "val_auc_roc",
    },
}


def run_final_gate():
    failures = []
    print("=" * 60)
    print("FINAL GATE -- Phase 2 P2 -- All Three Scripts")
    print("=" * 60)

    for script_name, expected_fields in EXPECTED.items():
        config_path = PHASE2_ARTIFACTS_DIR / f"{script_name}_run_config.json"
        print(f"\n--- {script_name} ---")

        if not config_path.exists():
            failures.append(f"{script_name}: {config_path.name} not found -- did this script finish?")
            print(f"  FAILED -- {config_path.name} not found")
            continue

        with open(config_path, "r", encoding="utf-8") as f:
            actual_config = json.load(f)

        for field, expected_val in expected_fields.items():
            actual_val = actual_config.get(field, "<missing>")
            if actual_val != expected_val:
                failures.append(
                    f"{script_name}.{field}: expected {expected_val!r}, got {actual_val!r}"
                )
                print(f"  FAILED -- {field}: expected {expected_val!r}, got {actual_val!r}")
            else:
                print(f"  OK -- {field}: {actual_val!r}")

    print("\n" + "=" * 60)
    if failures:
        print(f"FINAL GATE FAILED -- {len(failures)} issue(s):")
        for f in failures:
            print(f"  - {f}")
        print("=" * 60)
        raise RuntimeError(
            f"Final gate failed with {len(failures)} issue(s). Do not treat these "
            f"results as the locked Phase 6 baseline reference until resolved."
        )
    else:
        print("FINAL GATE PASSED -- all three scripts match the locked spec.")
        print("These numbers are now the fixed Phase 6 baseline reference. "
              "No further tuning on any of the three.")
        print("=" * 60)


# --- Usage ---
# run_final_gate()   # <-- call once, after all three scripts have finished

In [9]:
"""
00_08_build_models.txt
Phase 2 - P3 -- Section 0: Model Builders (split dropout decision)

Cluster 3 decision, derived from The_Methodology_v3.0.md + Implementation_Runbook_v1.0.md:

  - EfficientNet-B3 (Script 1, the image-only ablation) DOES get
    Dropout(0.5) before the final layer. Methodology 4.1 defines the
    ablation as "identical architecture, metadata branch and fusion
    mechanism removed entirely" -- only those two things are removed, so
    it inherits 4.4's classification head spec: "Single dense layer ->
    Dropout(0.5) -> 7-class Softmax."

  - ResNet50 (Script 2) and ViT-B/16 (Script 3) do NOT get dropout --
    plain Linear only. Methodology 8.3 / the Runbook frame these as "a
    clean, minimally-modified reference point," and the Runbook explicitly
    warns against "evening things out" to match the multimodal model's
    design, since that would blur what the baseline comparison is meant
    to isolate.

  - ResNet50 uses ImageNet1K_V1 weights (NOT V2) -- locked decision, for
    comparability with the literature baselines this thesis cites and to
    avoid confounding the "clean baseline" framing with a stronger,
    differently-trained backbone.
"""

from typing import cast

import torch.nn as nn
import torchvision.models as tvm

NUM_CLASSES = 7  # HAM10000 taxonomy


def build_efficientnet_b3(num_classes: int = NUM_CLASSES) -> nn.Module:
    """Script 1 -- image-only ablation. Dropout(0.5) -> Linear, per
    Methodology 4.1 + 4.4 (ablation inherits the multimodal model's
    classification head design)."""
    model = tvm.efficientnet_b3(weights=tvm.EfficientNet_B3_Weights.IMAGENET1K_V1)
    old_head = cast(nn.Linear, model.classifier[-1])
    model.classifier[-1] = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(old_head.in_features, num_classes),
    )
    return model


def build_resnet50(num_classes: int = NUM_CLASSES) -> nn.Module:
    """Script 2 -- clean, minimally-modified baseline. Plain Linear, no
    dropout, per Methodology 8.3 / Runbook's explicit warning against
    importing multimodal-model design choices into these baselines.
    ImageNet1K_V1 weights -- locked, not V2."""
    model = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
    old_head = cast(nn.Linear, model.fc)
    model.fc = nn.Linear(old_head.in_features, num_classes)
    return model


def build_vit_b16(num_classes: int = NUM_CLASSES) -> nn.Module:
    """Script 3 -- clean, minimally-modified baseline. Plain Linear, no
    dropout, same reasoning as ResNet50 above."""
    model = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1)
    old_head = cast(nn.Linear, model.heads.head)
    model.heads.head = nn.Linear(old_head.in_features, num_classes)
    return model

In [10]:
"""
01_00_dataset.txt
Phase 2 - P3 -- Section 1: Fold-Aware Dataset

Reuses Phase 1's own augment_image() (08_07) and normalize_image() (08_06)
VERBATIM -- these are already validated (pixel-for-pixel reproducibility
confirmed in Phase 1 Section 11), so Phase 2 doesn't reinvent them.

Handles all three scripts via a `resize_to` parameter:
  - EfficientNet-B3 / ResNet50 (Scripts 1/2): resize_to=None -- images are
    already saved at 300x300, used as-is.
  - ViT-B/16 (Script 3): resize_to=224 -- same cleaned 300x300 images,
    resized down. Order is: load -> resize to target square -> augment
    (train only) -> normalize. Augmentation operates on whatever square
    size it's given, so resizing first keeps its geometry simple and
    matches "same cleaned images, just resized down" from the spec --
    not a separate from-scratch pipeline.

CLASS_TO_IDX is fixed and alphabetical -- this exact mapping is what
Mechanism A (00_06) checks the loss-weight tensor against, so it must be
the SAME mapping used here and when building the loss weight tensor.
"""

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

BASE_DIR = Path(r"T:\Thesis - Development")
ARTIFACTS_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2"
FINAL_IMAGES_DIR = ARTIFACTS_DIR / "Final Output Phase 1" / "images"
CV_FOLD_PATH = ARTIFACTS_DIR / "cv_fold_assignments.csv"

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]  # alphabetical, fixed
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}


class HAM10000FoldDataset(Dataset):
    def __init__(self, fold_num: int, split: str, resize_to: int | None = None):
        """
        fold_num: 0-4, which fold is held out as validation.
        split: "train" (folds != fold_num) or "val" (fold == fold_num).
        resize_to: None (use 300x300 as saved) or 224 (ViT branch).
        """
        assert split in ("train", "val")
        self.split = split
        self.resize_to = resize_to

        cv_df = pd.read_csv(CV_FOLD_PATH)
        if split == "train":
            self.df = cv_df[cv_df["cv_fold"] != fold_num].reset_index(drop=True)
        else:
            self.df = cv_df[cv_df["cv_fold"] == fold_num].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row["image_id"]
        label = CLASS_TO_IDX[row["dx"]]

        img_path = FINAL_IMAGES_DIR / f"{image_id}.jpg"
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Failed to load image: {img_path}")

        if self.resize_to is not None:
            image = cv2.resize(image, (self.resize_to, self.resize_to), interpolation=cv2.INTER_AREA)

        if self.split == "train":
            image = augment_image(image)  # training-only, per Phase 1 Section 8/11

        normalized = normalize_image(image)  # BGR->RGB, /255, ImageNet mean/std -- both splits
        tensor = torch.from_numpy(normalized.transpose(2, 0, 1)).float()  # HWC -> CHW

        return tensor, label

    def get_labels(self):
        """Fast label access without loading images -- used by the balanced
        sampler (Script 1 only), which only needs the label list."""
        return self.df["dx"].map(CLASS_TO_IDX).values


# --- Reused verbatim from Phase 1 (08_06_normalization.txt) ---
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def normalize_image(image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_float = image_rgb.astype(np.float32) / 255.0
    normalized = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    return normalized


# --- Reused verbatim from Phase 1 (08_07_augmentation.txt) ---
AUGMENTATION_CONFIG = {
    "hflip_prob": 0.5,
    "vflip_prob": 0.5,
    "rotation_max_degrees": 180,
    "zoom_range": (0.9, 1.1),
    "brightness_range": (0.8, 1.2),
}


def augment_image(image, config=AUGMENTATION_CONFIG):
    h, w = image.shape[:2]
    out = image.copy()

    if np.random.random() < config["hflip_prob"]:
        out = cv2.flip(out, 1)
    if np.random.random() < config["vflip_prob"]:
        out = cv2.flip(out, 0)

    angle = np.random.uniform(-config["rotation_max_degrees"], config["rotation_max_degrees"])
    center = (w / 2, h / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    out = cv2.warpAffine(out, rotation_matrix, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    zoom_factor = np.random.uniform(*config["zoom_range"])
    new_h, new_w = int(round(h * zoom_factor)), int(round(w * zoom_factor))
    scaled = cv2.resize(out, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    if zoom_factor >= 1.0:
        # Zoomed in — centre-crop back down to original size
        row_start = (new_h - h) // 2
        col_start = (new_w - w) // 2
        out = scaled[row_start:row_start + h, col_start:col_start + w]
    else:
        # Zoomed out — pad back up to original size
        pad_h = h - new_h
        pad_w = w - new_w
        top, bottom = pad_h // 2, pad_h - pad_h // 2
        left, right = pad_w // 2, pad_w - pad_w // 2
        out = cv2.copyMakeBorder(scaled, top, bottom, left, right, borderType=cv2.BORDER_REFLECT_101)

    # Brightness
    brightness_factor = np.random.uniform(*config["brightness_range"])
    out = np.clip(out.astype(np.float32) * brightness_factor, 0, 255).astype(np.uint8)

    return out

In [11]:
"""
01_01_metrics.txt
Phase 2 - P3 -- Section 1: Per-Fold Metrics

Definitions follow Methodology 6.1's evaluation protocol table:
  - AUC-ROC: macro-averaged, one-vs-rest (the "headline" figure there)
  - F1, Sensitivity, Specificity: 6.1 specifies these per-class for
    Phase 6's demographic breakdown. Phase 2's baseline table only needs
    ONE number per metric per fold (spec: "Record results per fold --
    AUC-ROC, F1, sensitivity, specificity, accuracy" / "mean +/- SD across
    all 5 folds for each metric") -- INFERENCE: macro-averaging these
    three the same way AUC-ROC already is, for consistency. Flag if a
    different aggregation (e.g. weighted) was actually intended.
  - Accuracy: 6.1 explicitly notes "reported last, not primary" -- kept
    here as a single scalar, same as the other four.

Sensitivity = macro recall. Specificity has no direct sklearn multiclass
function -- computed per-class from the confusion matrix, then
macro-averaged, same pattern as the other three.
"""

import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, recall_score, roc_auc_score

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]  # must match dataset's CLASS_TO_IDX order


def compute_specificity_macro(y_true, y_pred, n_classes: int = 7) -> float:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    specificities = []
    total = cm.sum()
    for i in range(n_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        specificity_i = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(specificity_i)
    return float(np.mean(specificities))


def compute_fold_metrics(y_true, y_pred_probs, n_classes: int = 7) -> dict:
    """
    y_true: array of true integer labels, one per sample.
    y_pred_probs: array of shape (n_samples, n_classes), softmax probabilities.
    Returns: dict with auc_roc, f1, sensitivity, specificity, accuracy --
             all single floats, this fold's values.
    """
    y_pred_labels = np.argmax(y_pred_probs, axis=1)

    auc_roc = roc_auc_score(
        y_true, y_pred_probs, multi_class="ovr", average="macro",
        labels=list(range(n_classes)),
    )
    f1 = f1_score(y_true, y_pred_labels, average="macro", labels=list(range(n_classes)), zero_division=0)
    sensitivity = recall_score(y_true, y_pred_labels, average="macro", labels=list(range(n_classes)), zero_division=0)
    specificity = compute_specificity_macro(y_true, y_pred_labels, n_classes=n_classes)
    accuracy = float(np.mean(y_true == y_pred_labels))

    return {
        "auc_roc": float(auc_roc),
        "f1": float(f1),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "accuracy": accuracy,
    }


def summarize_across_folds(fold_metrics_list: list[dict]) -> dict:
    """fold_metrics_list: list of 5 dicts (one per fold), each from
    compute_fold_metrics(). Returns mean +/- SD for each metric."""
    summary = {}
    for key in ["auc_roc", "f1", "sensitivity", "specificity", "accuracy"]:
        values = [m[key] for m in fold_metrics_list]
        summary[key] = {"mean": float(np.mean(values)), "sd": float(np.std(values, ddof=1))}
    return summary

In [12]:
"""
01_02_train_fold.txt
Phase 2 - P3 -- Section 1: Generic Per-Fold Training Loop

Shared by all three scripts via parameters -- NOT a shared data-loading
setup with model-specific branches inside (the spec requires three fully
independent scripts for traceability). Each script imports this function
and calls it 5 times (once per fold) with its own build_model_fn,
resize_to, and use_balancing -- the function body itself has no
model-specific logic beyond what those parameters control.

num_workers=0 is a deliberate choice, not an oversight: DataLoader worker
processes get their own RNG state that the main process's
capture_rng_state()/restore_rng_state() (00_03) cannot see or restore, so
a power-cut resume could not honestly claim exact reproducibility of the
augmentation sequence with workers > 0. Data loading runs in the main
process instead, trading some throughput for a resume story that's
actually correct, not just approximately correct. Revisit if data loading
becomes the bottleneck (GPU diagnostic showed plenty of VRAM headroom,
so compute is unlikely to be the constraint).

bf16 AMP needs no GradScaler (unlike fp16) -- bf16 has fp32's exponent
range, so there's no underflow risk loss scaling exists to prevent.
"""

from pathlib import Path
from typing import Callable, Sequence, cast

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16

BASE_DIR = Path(r"T:\Thesis - Development")
ARTIFACTS_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2"
WEIGHTS_PATH = ARTIFACTS_DIR / "final_class_weights_per_fold.csv"
CV_FOLD_PATH = ARTIFACTS_DIR / "cv_fold_assignments.csv"


def build_balanced_sampler(labels, seed=RANDOM_SEED, generator_state=None):
    """Reused from Phase 1's 10_04_balanced_sampler.txt, verbatim logic.
    Built ONCE per fold, reused across all epochs -- the sampler's
    internal generator state advances with each DataLoader iteration, so
    repeated epochs naturally get different draws without rebuilding.

    generator_state: this sampler's torch.Generator is a separate object
    from the global RNGs capture_rng_state()/restore_rng_state() (00_03)
    cover, so a power-cut resume must restore ITS state explicitly too --
    otherwise a resumed fold would silently restart this generator from
    `seed` instead of continuing the draw sequence from where training
    stopped. Pass state["sampler_generator_state"] from the resume
    checkpoint to continue exactly; leave None for a fresh fold."""
    labels_series = pd.Series(labels)
    class_counts = labels_series.value_counts()
    class_weight_map = 1.0 / class_counts
    sample_weights = labels_series.map(class_weight_map).values

    generator = torch.Generator()
    if generator_state is not None:
        generator.set_state(generator_state)
    else:
        generator.manual_seed(seed)

    return WeightedRandomSampler(
        # cast: PyTorch's stub declares weights: Sequence[float], but the real
        # implementation (and PyTorch's own docs) accept a Tensor directly --
        # a stub/runtime mismatch, not a type error.
        weights=cast(Sequence[float], torch.as_tensor(sample_weights, dtype=torch.double)),
        num_samples=len(labels_series),
        replacement=True,
        generator=generator,
    )


def build_weighted_criterion(fold_num: int, class_to_idx: dict) -> nn.CrossEntropyLoss:
    """Script 1 only. Loads Phase 1's saved effective-number weights
    (final_class_weights_per_fold.csv) for this fold, builds the weight
    tensor IN class_to_idx ORDER -- this exact ordering is what
    Mechanism A (00_06) checks against, so it must match."""
    saved_weights = pd.read_csv(WEIGHTS_PATH, index_col="dx")
    fold_col = str(fold_num)
    weight_tensor = torch.zeros(len(class_to_idx), dtype=torch.float32)
    for cls, idx in class_to_idx.items():
        weight_tensor[idx] = float(cast(float, saved_weights.loc[cls, fold_col]))
    return nn.CrossEntropyLoss(weight=weight_tensor.to(DEVICE))


def train_fold(
    fold_num: int,
    build_model_fn: Callable[[], nn.Module],
    resize_to: int | None,
    use_balancing: bool,
    batch_size: int,
    out_dir: Path,
    class_to_idx: dict,
    max_epochs: int = 100,
    patience: int = 10,
    lr: float = 1e-4,
    weight_decay: float = 1e-5,
    run_mechanism_checks: bool = False,
) -> dict:
    """
    Returns this fold's final metrics dict (from compute_fold_metrics,
    computed with the BEST checkpoint's weights, not the last epoch's).

    Requires the following already defined earlier in the SAME notebook
    (paste order, not imports -- matches the Phase 1 notebook convention):
      HAM10000FoldDataset (01_00), compute_fold_metrics (01_01),
      load_resume_state / save_resume_state / save_best_weights /
      restore_rng_state (00_03), verify_mechanism_a / verify_mechanism_b
      (00_06, only reached when run_mechanism_checks=True).
    """
    set_all_seeds(RANDOM_SEED)  # first line, per fold -- 00_02

    out_dir.mkdir(parents=True, exist_ok=True)
    resume_path = out_dir / f"fold_{fold_num}_resume.pt"
    best_path = out_dir / f"fold_{fold_num}_best.pt"

    # Loaded up front (not after building the sampler, as before) -- the
    # balanced sampler's own generator state, if any, has to be restored
    # WHILE the sampler is being built, not after.
    state = load_resume_state(resume_path)

    train_ds = HAM10000FoldDataset(fold_num, "train", resize_to=resize_to)
    val_ds = HAM10000FoldDataset(fold_num, "val", resize_to=resize_to)

    if use_balancing:
        saved_generator_state = state["sampler_generator_state"] if state is not None else None
        sampler = build_balanced_sampler(
            train_ds.get_labels(), seed=RANDOM_SEED, generator_state=saved_generator_state
        )
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
        criterion = build_weighted_criterion(fold_num, class_to_idx)
    else:
        sampler = None  # nothing sampler-specific to checkpoint -- shuffle=True uses the
                        # global torch RNG, already covered by capture_rng_state()
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
        criterion = nn.CrossEntropyLoss()

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    if run_mechanism_checks:  # verify_mechanism_a/b -- defined earlier, 00_06
        cv_df = pd.read_csv(CV_FOLD_PATH)
        verify_mechanism_a(criterion, cv_df, fold_num, class_to_idx)
        verify_mechanism_b(train_loader)

    model = build_model_fn().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if state is not None:
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        for opt_state in optimizer.state.values():  # move optimizer tensors back to DEVICE
            for k, v in opt_state.items():
                if torch.is_tensor(v):
                    opt_state[k] = v.to(DEVICE)
        restore_rng_state(state["rng_state"])
        best_metric = state["best_metric"]
        early_stop_counter = state["early_stop_counter"]
        if state.get("finished", False):
            # Already early-stopped in a previous execution of this script --
            # skip training entirely (empty range) and fall through to final
            # metrics below, using the saved best checkpoint. Without this
            # check a re-run would silently resume training past a
            # legitimate stop.
            start_epoch = max_epochs
            print(f"Fold {fold_num}: already finished (early-stopped in a previous run) -- "
                  f"skipping straight to final metrics from the saved best checkpoint.")
        else:
            start_epoch = state["epoch"] + 1
            print(f"Fold {fold_num}: resuming from epoch {start_epoch}")
    else:
        start_epoch = 0
        best_metric = -float("inf")
        early_stop_counter = 0
        print(f"Fold {fold_num}: starting fresh")

    for epoch in range(start_epoch, max_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(images)
                loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    logits = model(images)
                probs = torch.softmax(logits.float(), dim=1)
                all_probs.append(probs.cpu().numpy())
                all_labels.append(labels.numpy())

        y_pred_probs = np.concatenate(all_probs)
        y_true = np.concatenate(all_labels)
        val_metrics = compute_fold_metrics(y_true, y_pred_probs, n_classes=len(class_to_idx))
        val_auc_roc = val_metrics["auc_roc"]

        print(f"Fold {fold_num}, epoch {epoch}: val AUC-ROC={val_auc_roc:.4f} "
              f"(best={best_metric:.4f}, patience={early_stop_counter}/{patience})")

        # Decision update now happens BEFORE the checkpoint write (previously
        # after) -- so the saved epoch/best_metric/early_stop_counter always
        # reflect THIS epoch's actual outcome, not the prior epoch's, and
        # "finished" below can be computed and persisted truthfully.
        if val_auc_roc > best_metric:
            best_metric = val_auc_roc
            early_stop_counter = 0
            save_best_weights(best_path, model)
        else:
            early_stop_counter += 1

        just_finished = early_stop_counter >= patience

        # sampler is None for Scripts 2/3 -- no sampler-specific generator to save.
        # cast: sampler.generator is typed Generator | None in the stub, but
        # build_balanced_sampler() always constructs it with a real Generator --
        # it's only ever None here because the whole sampler is None.
        sampler_generator_state = (
            cast(torch.Generator, sampler.generator).get_state() if sampler is not None else None
        )
        save_resume_state(resume_path, epoch, model, optimizer, best_metric, early_stop_counter,
                           sampler_generator_state=sampler_generator_state, finished=just_finished)

        if just_finished:
            print(f"Fold {fold_num}: early stopping at epoch {epoch}")
            break

    # --- Final metrics, using the BEST checkpoint, not the last epoch's weights ---
    best_state = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_state["model_state_dict"])
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(images)
            probs = torch.softmax(logits.float(), dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())

    y_pred_probs = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)
    final_metrics = compute_fold_metrics(y_true, y_pred_probs, n_classes=len(class_to_idx))
    print(f"Fold {fold_num} FINAL (best checkpoint): {final_metrics}")
    return final_metrics


# --- set_all_seeds, reused from 00_02 (duplicated here so this file has
#     no import-order dependency on notebook cell execution order) ---
import random


def set_all_seeds(seed: int = RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    import os
    os.environ["PYTHONHASHSEED"] = str(seed)

In [13]:
"""
01_03_script1_efficientnet_b3.txt
Phase 2 - P3 -- Script 1: EfficientNet-B3 (Image-Only Ablation)

Requires already defined earlier in this notebook (paste order):
  00_02 (set_all_seeds), 00_03 (checkpoint utils), 00_04 (preflight --
  run once, before this cell), 00_06 (Mechanism A/B), 00_08
  (build_efficientnet_b3), 01_00 (Dataset, CLASS_TO_IDX), 01_01 (metrics),
  01_02 (train_fold).

Mechanism A/B verification runs on fold 0 ONLY (run_mechanism_checks=True)
-- confirming the wiring is correct once is sufficient; it's a structural
check on how the fold is built, not something that could pass on fold 0
and silently fail on fold 1-4, since every fold goes through the same
build_weighted_criterion() / build_balanced_sampler() code path.
"""

from pathlib import Path

OUT_DIR = PHASE2_ARTIFACTS_DIR / "script1_efficientnet_b3"  # PHASE2_ARTIFACTS_DIR from 00_05
BATCH_SIZE = 32  # reverted from 64 -- restores the batch size validated in Phase 2-P1
                 # (0.947-0.952 mean AUC-ROC); 64 caused 2/5 folds to early-stop
                 # prematurely under patience=10 (see decisions-and-learnings)

fold_metrics = []
for fold_num in range(5):
    metrics = train_fold(
        fold_num=fold_num,
        build_model_fn=build_efficientnet_b3,
        resize_to=None,  # already 300x300 as saved
        use_balancing=True,
        batch_size=BATCH_SIZE,
        out_dir=OUT_DIR,
        class_to_idx=CLASS_TO_IDX,
        run_mechanism_checks=(fold_num == 0),
    )
    fold_metrics.append(metrics)

summary = summarize_across_folds(fold_metrics)
print("\n" + "=" * 60)
print("SCRIPT 1 -- EfficientNet-B3 -- Summary Across 5 Folds")
print("=" * 60)
for metric_name, stats in summary.items():
    print(f"  {metric_name}: {stats['mean']:.4f} +/- {stats['sd']:.4f}")

save_run_config("script1_efficientnet_b3", {
    "model": "efficientnet_b3",
    "resolution": 300,
    "balancing_enabled": True,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "weight_decay": 1e-5,
    "learning_rate": 1e-4,
    "early_stopping_patience": 10,
    "max_epochs": 100,
    "checkpoint_policy": "best_epoch_auc_roc",
    "monitor_metric": "val_auc_roc",
    "amp_dtype": "bfloat16",
    "random_seed": RANDOM_SEED,
    "dropout_head": True,
    "fold_metrics": fold_metrics,
    "summary": summary,
})

C:\Users\primu\AppData\Local\Temp\ipykernel_14988\700060210.py:73: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights=cast(Sequence[float], torch.as_tensor(sample_weights, dtype=torch.double)),


--- Mechanism A: loss-weight wiring check, fold 0 ---
  OK -- criterion.weight matches expected effective-number weights, correct class order, all 7 classes
--- Mechanism B: balanced-sampler wiring check (300 batches) ---
  Ideal balanced target: 14.29% per class
1    14.65
0    14.52
3    14.38
2    14.35
5    14.32
4    14.17
6    13.62
  OK -- max deviation from ideal target: 0.67pp (tolerance: 3.0pp)
Fold 0: starting fresh
Fold 0, epoch 0: val AUC-ROC=0.7752 (best=-inf, patience=0/10)
Fold 0, epoch 1: val AUC-ROC=0.8523 (best=0.7752, patience=0/10)
Fold 0, epoch 2: val AUC-ROC=0.8819 (best=0.8523, patience=0/10)
Fold 0, epoch 3: val AUC-ROC=0.8794 (best=0.8819, patience=0/10)
Fold 0, epoch 4: val AUC-ROC=0.8305 (best=0.8819, patience=1/10)
Fold 0, epoch 5: val AUC-ROC=0.8593 (best=0.8819, patience=2/10)
Fold 0, epoch 6: val AUC-ROC=0.8762 (best=0.8819, patience=3/10)
Fold 0, epoch 7: val AUC-ROC=0.8181 (best=0.8819, patience=4/10)
Fold 0, epoch 8: val AUC-ROC=0.8419 (best=0.8819, p

In [14]:
"""
01_04_script2_resnet50.txt
Phase 2 - P3 -- Script 2: ResNet50

Identical structure to Script 1, except: no class-balancing (standard
unweighted loss, natural batch sampling -- use_balancing=False), and
build_model_fn=build_resnet50 (plain Linear head, ImageNet1K_V1 weights).

Requires the same earlier cells as Script 1, EXCEPT 00_06 (Mechanism A/B
is Script 1-only -- there's no balancing here to verify).
"""

OUT_DIR = PHASE2_ARTIFACTS_DIR / "script2_resnet50"  # PHASE2_ARTIFACTS_DIR from 00_05
BATCH_SIZE = 32  # reverted from 64 -- restores the batch size validated in Phase 2-P1
                 # (0.947-0.952 mean AUC-ROC); 64 caused 2/5 folds to early-stop
                 # prematurely under patience=10 (see decisions-and-learnings)

fold_metrics = []
for fold_num in range(5):
    metrics = train_fold(
        fold_num=fold_num,
        build_model_fn=build_resnet50,
        resize_to=None,  # already 300x300 as saved
        use_balancing=False,
        batch_size=BATCH_SIZE,
        out_dir=OUT_DIR,
        class_to_idx=CLASS_TO_IDX,
        run_mechanism_checks=False,
    )
    fold_metrics.append(metrics)

summary = summarize_across_folds(fold_metrics)
print("\n" + "=" * 60)
print("SCRIPT 2 -- ResNet50 -- Summary Across 5 Folds")
print("=" * 60)
for metric_name, stats in summary.items():
    print(f"  {metric_name}: {stats['mean']:.4f} +/- {stats['sd']:.4f}")

save_run_config("script2_resnet50", {
    "model": "resnet50",
    "resolution": 300,
    "balancing_enabled": False,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "weight_decay": 1e-5,
    "learning_rate": 1e-4,
    "early_stopping_patience": 10,
    "max_epochs": 100,
    "checkpoint_policy": "best_epoch_auc_roc",
    "monitor_metric": "val_auc_roc",
    "amp_dtype": "bfloat16",
    "random_seed": RANDOM_SEED,
    "dropout_head": False,
    "pretrained_weights": "IMAGENET1K_V1",
    "fold_metrics": fold_metrics,
    "summary": summary,
})

Fold 0: starting fresh
Fold 0, epoch 0: val AUC-ROC=0.9454 (best=-inf, patience=0/10)
Fold 0, epoch 1: val AUC-ROC=0.9475 (best=0.9454, patience=0/10)
Fold 0, epoch 2: val AUC-ROC=0.9412 (best=0.9475, patience=0/10)
Fold 0, epoch 3: val AUC-ROC=0.9495 (best=0.9475, patience=1/10)
Fold 0, epoch 4: val AUC-ROC=0.9636 (best=0.9495, patience=0/10)
Fold 0, epoch 5: val AUC-ROC=0.9449 (best=0.9636, patience=0/10)
Fold 0, epoch 6: val AUC-ROC=0.9471 (best=0.9636, patience=1/10)
Fold 0, epoch 7: val AUC-ROC=0.9543 (best=0.9636, patience=2/10)
Fold 0, epoch 8: val AUC-ROC=0.9130 (best=0.9636, patience=3/10)
Fold 0, epoch 9: val AUC-ROC=0.9476 (best=0.9636, patience=4/10)
Fold 0, epoch 10: val AUC-ROC=0.9464 (best=0.9636, patience=5/10)
Fold 0, epoch 11: val AUC-ROC=0.9538 (best=0.9636, patience=6/10)
Fold 0, epoch 12: val AUC-ROC=0.9456 (best=0.9636, patience=7/10)
Fold 0, epoch 13: val AUC-ROC=0.9590 (best=0.9636, patience=8/10)
Fold 0, epoch 14: val AUC-ROC=0.9638 (best=0.9636, patience=9/10)

In [15]:
"""
01_05_script3_vit_b16.txt
Phase 2 - P3 -- Script 3: ViT-B/16

Identical structure to Script 2, except: resize_to=224 (same cleaned
300x300 images, resized down -- NOT a separate from-scratch pipeline),
and build_model_fn=build_vit_b16.
"""

OUT_DIR = PHASE2_ARTIFACTS_DIR / "script3_vit_b_16"  # PHASE2_ARTIFACTS_DIR from 00_05
BATCH_SIZE = 32  # reverted from 64 -- restores the batch size validated in Phase 2-P1
                 # (0.947-0.952 mean AUC-ROC); 64 caused 2/5 folds to early-stop
                 # prematurely under patience=10 (see decisions-and-learnings)

fold_metrics = []
for fold_num in range(5):
    metrics = train_fold(
        fold_num=fold_num,
        build_model_fn=build_vit_b16,
        resize_to=224,  # same cleaned images, resized down from 300x300
        use_balancing=False,
        batch_size=BATCH_SIZE,
        out_dir=OUT_DIR,
        class_to_idx=CLASS_TO_IDX,
        run_mechanism_checks=False,
    )
    fold_metrics.append(metrics)

summary = summarize_across_folds(fold_metrics)
print("\n" + "=" * 60)
print("SCRIPT 3 -- ViT-B/16 -- Summary Across 5 Folds")
print("=" * 60)
for metric_name, stats in summary.items():
    print(f"  {metric_name}: {stats['mean']:.4f} +/- {stats['sd']:.4f}")

save_run_config("script3_vit_b_16", {
    "model": "vit_b_16",
    "resolution": 224,
    "balancing_enabled": False,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "weight_decay": 1e-5,
    "learning_rate": 1e-4,
    "early_stopping_patience": 10,
    "max_epochs": 100,
    "checkpoint_policy": "best_epoch_auc_roc",
    "monitor_metric": "val_auc_roc",
    "amp_dtype": "bfloat16",
    "random_seed": RANDOM_SEED,
    "dropout_head": False,
    "pretrained_weights": "IMAGENET1K_V1",
    "fold_metrics": fold_metrics,
    "summary": summary,
})

# --- After all three scripts have finished: run the final gate (00_07) ---
# run_final_gate()

Fold 0: starting fresh
Fold 0, epoch 0: val AUC-ROC=0.9456 (best=-inf, patience=0/10)
Fold 0, epoch 1: val AUC-ROC=0.9604 (best=0.9456, patience=0/10)
Fold 0, epoch 2: val AUC-ROC=0.9599 (best=0.9604, patience=0/10)
Fold 0, epoch 3: val AUC-ROC=0.9646 (best=0.9604, patience=1/10)
Fold 0, epoch 4: val AUC-ROC=0.9579 (best=0.9646, patience=0/10)
Fold 0, epoch 5: val AUC-ROC=0.9621 (best=0.9646, patience=1/10)
Fold 0, epoch 6: val AUC-ROC=0.9603 (best=0.9646, patience=2/10)
Fold 0, epoch 7: val AUC-ROC=0.9610 (best=0.9646, patience=3/10)
Fold 0, epoch 8: val AUC-ROC=0.9640 (best=0.9646, patience=4/10)
Fold 0, epoch 9: val AUC-ROC=0.9635 (best=0.9646, patience=5/10)
Fold 0, epoch 10: val AUC-ROC=0.9603 (best=0.9646, patience=6/10)
Fold 0, epoch 11: val AUC-ROC=0.9589 (best=0.9646, patience=7/10)
Fold 0, epoch 12: val AUC-ROC=0.9563 (best=0.9646, patience=8/10)
Fold 0, epoch 13: val AUC-ROC=0.9500 (best=0.9646, patience=9/10)
Fold 0: early stopping at epoch 13
Fold 0 FINAL (best checkpoint)

In [10]:
run_final_gate()

FINAL GATE -- Phase 2 P2 -- All Three Scripts

--- script1_efficientnet_b3 ---
  OK -- model: 'efficientnet_b3'
  OK -- resolution: 300
  OK -- balancing_enabled: True
  OK -- optimizer: 'AdamW'
  OK -- weight_decay: 1e-05
  OK -- checkpoint_policy: 'best_epoch_auc_roc'
  OK -- monitor_metric: 'val_auc_roc'

--- script2_resnet50 ---
  OK -- model: 'resnet50'
  OK -- resolution: 300
  OK -- balancing_enabled: False
  OK -- optimizer: 'AdamW'
  OK -- weight_decay: 1e-05
  OK -- checkpoint_policy: 'best_epoch_auc_roc'
  OK -- monitor_metric: 'val_auc_roc'

--- script3_vit_b_16 ---
  OK -- model: 'vit_b_16'
  OK -- resolution: 224
  OK -- balancing_enabled: False
  OK -- optimizer: 'AdamW'
  OK -- weight_decay: 1e-05
  OK -- checkpoint_policy: 'best_epoch_auc_roc'
  OK -- monitor_metric: 'val_auc_roc'

FINAL GATE PASSED -- all three scripts match the locked spec.
These numbers are now the fixed Phase 6 baseline reference. No further tuning on any of the three.
